# PhiState-MoE: compare 10K vs 10,250, then train to 50K on CUDA

**What this does:** (1) Restore the original corpus/tokenizer and verify the 10,250-step checkpoint in your private Google Drive; (2) compare **10K vs 10,250 actual generations** using identical coding/chat prompts, greedy decoding and seeded sampling; (3) start an **attended, restart-safe NVIDIA CUDA run toward 50,000 TOTAL optimizer steps**. After each completed ~500-step segment, the best and resumable optimizer checkpoints are verified in alternating Google Drive slots. Separate milestone pairs are retained at 20K, 30K, 40K and 50K.

**Run this in Google Colab, not Codespaces.** Choose **Runtime → Change runtime type → L4 GPU** (T4/A100 can work), then **Runtime → Run all**. This starts prolonged CUDA training after the comparison, consuming compute units. Do not deliberately disconnect Colab during a segment.

After a Colab reset, **Run all again**. The notebook restores the original dataset from private Drive and the runner resumes from the last *completed, verified* segment. It cannot run while Colab is disconnected. Your 10K and 10,250 source checkpoints are protected and never overwritten. Encrypted GitHub Release archives are used only when needed.

**Scientific limits:** This remains a 38M-parameter research language model; decreasing validation BPT does not establish code correctness or conversational competence. The comparison displays actual generated text and basic Python syntax indicators, not a comprehensive benchmark. Colab session timeouts, hardware availability or exhausted credits can interrupt the run; progress is durable only through the last verified segment.



In [ ]:
# 1 — Connect GPU, clone/pull source, mount Drive, restore & verify the original corpus.
import sys, json, shutil, subprocess, importlib.util
from pathlib import Path
from getpass import getpass
import torch

assert torch.cuda.is_available(), "Choose Runtime > Change runtime type > L4 GPU and reconnect."
print("CUDA:", torch.cuda.get_device_name(0), flush=True)

from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/phistate_l4")
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / "quantoniumoslinux"
LAB = REPO / "runtime/experiments/phi_runtime_lab"
DRIVE = Path("/content/drive/MyDrive/Quantonium/PhiState-L4")
DRIVE.mkdir(parents=True, exist_ok=True)
RUN_DRIVE = DRIVE / "50k"

if (REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
elif REPO.exists():
    raise RuntimeError(f"Partial clone at {REPO}. Inspect before retrying.")
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Anharmoniclabs/quantoniumoslinux.git", str(REPO)], check=True)
for filename in ("moe_train.py", "moe_model.py", "compare_phistate_generations.py",
                 "colab_50k_runner.py", "pretrain_continue_10k.py"):
    assert (LAB / filename).is_file(), f"Missing GitHub file {filename}"
if importlib.util.find_spec("sentencepiece") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=True)
if shutil.which("7z") is None:
    subprocess.run(["apt-get", "-qq", "install", "-y", "7zip"], check=True)

RELEASE = ("https://github.com/Anharmoniclabs/quantoniumoslinux/"
           "releases/download/phistate-colab-probe-v1/")
ARCHIVE_SIZES = {"phi_dataset_rekey.7z": 32838913, "phi_final10k.7z": 141204483}
ARCHIVES = ROOT / "archives"
ARCHIVES.mkdir(exist_ok=True)

def restore_encrypted(name, destination, expected):
    if expected.is_file():
        print("Already restored:", expected, flush=True)
        return
    destination.mkdir(parents=True, exist_ok=True)
    archive = ARCHIVES / name
    if archive.is_file() and archive.stat().st_size != ARCHIVE_SIZES[name]:
        archive.unlink()
    if not archive.is_file():
        pending = archive.with_name(archive.name + ".download")
        pending.unlink(missing_ok=True)
        print("Downloading", name, flush=True)
        subprocess.run(["wget", "-q", "--show-progress", "-O", str(pending),
                        RELEASE + name], check=True)
        if pending.stat().st_size != ARCHIVE_SIZES[name]:
            raise RuntimeError(f"Download incomplete: {name}")
        pending.rename(archive)
    password = getpass(f"7-Zip password for {name} (never share it): ")
    try:
        check = subprocess.run(["7z", "t", "-p" + password, str(archive)],
                               capture_output=True, text=True)
        if check.returncode:
            raise RuntimeError(f"Wrong archive password for {name}. Re-enter the password used when creating THIS archive.")
        subprocess.run(["7z", "x", "-y", "-p" + password,
                        "-o" + str(destination), str(archive)],
                       capture_output=True, text=True, check=True)
    finally:
        del password
    if not expected.is_file():
        raise FileNotFoundError(f"Expected extracted path missing: {expected}")

LATEST_10250 = DRIVE / "phi_10250.last.pt"
BEST_10250 = DRIVE / "phi_10250.pt"
assert LATEST_10250.is_file() and BEST_10250.is_file(), (
    "Missing saved 10,250-step checkpoints in Google Drive /Quantonium/PhiState-L4"
)

DATA = ROOT / "dataset_transfer" / "hf_coding_knowledge_v2"
DRIVE_DATA = DRIVE / "data" / "hf_coding_knowledge_v2"
if not (DATA / "manifest.json").is_file():
    if (DRIVE_DATA / "manifest.json").is_file():
        print("Restoring original tokenized corpus from private Drive", flush=True)
        shutil.copytree(DRIVE_DATA, DATA, dirs_exist_ok=True)
    else:
        restore_encrypted("phi_dataset_rekey.7z", DATA.parent, DATA / "manifest.json")

manifest = json.loads((DATA / "manifest.json").read_text())
EXPECTED_FINGERPRINT = "29e4997527c221f48180d30f5d5d3245d0b5476e69a3b735d5b0895204982474"
assert manifest["data_fingerprint"] == EXPECTED_FINGERPRINT
assert (DATA / "tokenizer.model").is_file()
sys.path.insert(0, str(LAB))
from moe_train import load_data
_, splits, _ = load_data(str(DATA))
print("Corpus splits verified:", sorted(splits))
del splits

latest = torch.load(LATEST_10250, map_location="cpu", weights_only=True)
assert latest["step"] == 10250, "Wrong 10,250-step checkpoint"
assert latest["data_fingerprint"] == EXPECTED_FINGERPRINT
assert latest["config"]["vocab"] == manifest["vocab"]
assert "optimizer_state" in latest
print("10,250 optimizer state VERIFIED; validation BPT:", latest["best_validation_bpt"])
del latest

if not (DRIVE_DATA / "manifest.json").is_file():
    print("Saving reusable original corpus to private Google Drive (about 79 MiB)...", flush=True)
    DRIVE_DATA.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DATA, DRIVE_DATA, dirs_exist_ok=True)
    assert (DRIVE_DATA / "manifest.json").is_file()

# Verify mounted Drive supports the atomic pointer rename needed by the runner.
RUN_DRIVE.mkdir(parents=True, exist_ok=True)
probe = RUN_DRIVE / ".rename_check.pending"
probe_done = RUN_DRIVE / ".rename_check.ok"
probe.write_text("ok")
probe.replace(probe_done)
assert probe_done.read_text() == "ok"
probe_done.unlink()
print("GPU, source, corpus, optimizer, persistent dataset and Drive writes: READY", flush=True)


In [ ]:
# 2 — Compare 10K and 10,250 on identical coding/chat prompts.
# Saves a readable Markdown report and structured JSON privately in Google Drive.
import json, subprocess, sys, shutil
from pathlib import Path

QUALITY_DIR = DRIVE / "comparison_10k_vs_10250"
QUALITY_MD = QUALITY_DIR / "quality_10k_vs_10250.md"
if QUALITY_MD.is_file():
    print("Existing comparison in private Google Drive:", QUALITY_MD)
    print(QUALITY_MD.read_text(encoding="utf8")[:18000])
else:
    BASELINE_DRIVE = DRIVE / "phi_10000.pt"
    if not BASELINE_DRIVE.is_file():
        restored = ROOT / "final10k" / "resume.last.pt"
        if not restored.is_file():
            restore_encrypted("phi_final10k.7z", restored.parent, restored)
        ck = torch.load(restored, map_location="cpu", weights_only=True)
        assert ck["step"] == 10000 and ck["data_fingerprint"] == EXPECTED_FINGERPRINT
        del ck
        shutil.copy2(restored, BASELINE_DRIVE)
        print("Saved 10K comparison baseline to private Drive")
    QUALITY_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        sys.executable, "-u", str(LAB / "compare_phistate_generations.py"),
        "--baseline", str(BASELINE_DRIVE),
        "--candidate", str(BEST_10250),
        "--tokenizer", str(DATA / "tokenizer.model"),
        "--output-dir", str(QUALITY_DIR),
        "--new-tokens", "80", "--threads", "2", "--seed", "20261027",
    ], check=True)
    print("\n========== ACTUAL GENERATIONS ==========\n")
    print(QUALITY_MD.read_text(encoding="utf8")[:18000])
print("Comparison ready. Lower validation BPT alone is not proof of coherent answers.")


In [ ]:
# 3 — BEGIN / RESUME CUDA PRETRAINING TO 50,000 TOTAL STEPS.
# 'Runtime > Run all' executes this cell after the output comparison.
# The runner saves and verifies every completed ~500-step segment to private Drive.
# On Colab interruption, open this notebook and Run all again.
import subprocess, sys
from pathlib import Path

TARGET_STEPS = 50000  # TOTAL optimizer steps, not 50K additional steps
CHUNK_STEPS = 500     # At most an unfinished segment is lost on disconnect
MAX_CHUNKS = 0       # 0 = run toward target; 1 = run a single segment

command = [
    sys.executable, "-u", str(LAB / "colab_50k_runner.py"),
    "--lab", str(LAB), "--data-dir", str(DATA),
    "--initial-last", str(LATEST_10250),
    "--initial-best", str(BEST_10250),
    "--drive-dir", str(RUN_DRIVE),
    "--target-steps", str(TARGET_STEPS),
    "--chunk-steps", str(CHUNK_STEPS),
    "--eval-batches", "16", "--threads", "2",
    "--max-chunks", str(MAX_CHUNKS),
]
print("TRAINING: 10,250 -> 50,000 on", torch.cuda.get_device_name(0), flush=True)
print("Persistent progress:", RUN_DRIVE / "progress.json", flush=True)
subprocess.run(command, check=True)


In [ ]:
# 4 — Inspect the latest verified checkpoint.
import json
from pathlib import Path
marker = RUN_DRIVE / "progress.json"
if not marker.is_file():
    print("No completed segment yet; source 10,250 checkpoint remains protected.")
else:
    progress = json.loads(marker.read_text())
    print("SAVED STEP:", progress["step"], "/", progress["target_steps"])
    print("BEST VALIDATION BPT:", progress["best_validation_bpt"])
    print("LAST SESSION TOKENS/SEC:", progress.get("last_session_tokens_per_second"))
    print("ACTIVE RESUME SLOT:", progress["active_slot"])
    print("PROGRESS FILE:", marker)
